# 7장 — 개체명 중의성 해소

| | |
| --- | --- |
| **책** | Knowledge Graphs and LLMs in Action **7장** |
| **해설 원본** | [07_named_entity_disambiguation_ko_explained.md](../../07_named_entity_disambiguation_ko_explained.md) |
| **이 폴더** | [tasks.py](./tasks.py) · [importer](./importer) · [analysis](./analysis) · [Readme.md](./Readme.md) |
| **원서 저장소** | [ch09](https://github.com/alenegro81/knowledge-graphs-and-llms-in-action/tree/main/chapters/ch09) |
| **리스팅 번호** | 오프셋 **+0** — 책 번호 = 파일 번호 − 0 |

---

TODO(agent): 이 장이 앞 장과 무엇이 다른지, 왜 이 순서인지 2\~3문장.

| 절 | 내용 | 실습 |
| --- | --- | --- |
| 7.1 | From recognition to disambiguation — 인식에서 중의성 해소로 | |
| 7.2 | Understanding named entity disambiguation — 개체명 중의성 해소 이해하기 | |
| 7.3 | Domain-based NED and LLMs — 도메인 기반 NED와 LLM | |
| 7.4 | Business and domain understanding — 비즈니스와 도메인 이해 | |
| 7.5 | Understanding the data — 데이터 이해하기 | |
| 7.6 | Building a SoHO knowledge graph — SoHO 지식 그래프 구축하기 | |
| 7.7 | KG-based use cases — KG 기반 활용 사례 | |
| 7.7 | KG-based use cases — KG 기반 활용 사례 (이어서) | |

> **한 문장 요약** — TODO(agent)


---
## 실행 환경

교재 루트의 `.venv`(Python 3.10) 커널에서 돌아간다. 처음이면 교재 루트에서
[`setup_env.py`](../../../setup_env.py) 를 한 번 실행한다.

```bash
python3 setup_env.py
```

TODO(agent): 이 장이 필요한 외부 서비스(Neo4j 에디션·플러그인·API 키 등)와 그 이유.
에디션 제약이나 플러그인 충돌이 있으면 반드시 적는다 — 없으면 스터디원이 막힌다.
오래 걸리는 단계는 실측 시간과 용량을 적는다.


In [ ]:
"""환경 점검 — 이 셀이 통과하면 이후 모든 셀을 돌릴 수 있다."""
from pathlib import Path

from studykit import config, cypher

STUDY = config.load()     # 위로 올라가며 study.toml 을 찾는다
HERE = Path.cwd()         # 주피터는 노트북 폴더를 cwd 로 둔다

assert (HERE / "analysis").is_dir(), f"cwd 가 챕터 폴더가 아니다: {HERE}"

print("교재      :", STUDY.title)

# TODO(agent): 이 장이 요구하는 것을 검사하고, 실패 시 무엇을 하라고 알려라.
#   예) Neo4j 연결·에디션·플러그인, API 키 존재, 필요한 파이썬 패키지


In [ ]:
"""리스팅 접근 헬퍼.

리스팅 번호 대응은 study.toml 이 정본이다. 여기에 하드코딩하면 값이 두 곳에 생겨
갈라진다. 최종 교재가 코드 없는 리스팅(프롬프트 등)을 끼워넣으면 단일 오프셋으로는
표현되지 않으므로 resolve_listing() 을 쓴다.
"""
import time

REPO_DIR = "ch07"


def source_of(book_no: str) -> str:
    """책 리스팅 번호로 원문을 가져온다."""
    kind, value = STUDY.resolve_listing(REPO_DIR, book_no)
    if kind != "repo":
        raise KeyError(
            f"책 {book_no} 은 저장소에 코드 파일이 없다 (source={kind}). "
            f"해설판 본문을 실은 마크다운 셀을 보라."
        )
    return cypher.read(value, HERE / "listings").strip()


def show(book_no: str) -> None:
    kind, value = STUDY.resolve_listing(REPO_DIR, book_no)
    origin = f"파일 {value}" if kind == "repo" else "해설판 본문"
    print(f"── 책 Listing {book_no}  ({origin}) ──")
    print(source_of(book_no) if kind == "repo" else "(아래 마크다운 셀에 원문을 실었다)")


def statements(book_no: str) -> list:
    """주석만으로 된 조각은 버린다."""
    out = []
    for chunk in source_of(book_no).split(";"):
        lines = [l for l in chunk.strip().splitlines() if l.strip()]
        if lines and not all(l.strip().startswith("#") for l in lines):
            out.append("\n".join(l for l in lines if not l.strip().startswith("#")).strip())
    return out


def run(book_no: str, db: str = "neo4j", limit: int = 5, **params):
    """책 리스팅을 실행하고 결과 일부를 보여준다."""
    started = time.time()
    rows = []
    with cypher.driver() as drv, drv.session(database=db) as s:
        for statement in statements(book_no):
            rows = [r.data() for r in s.run(statement, **params)]
    print(f"책 {book_no} [{db}]: {time.time() - started:.1f}초, {len(rows)}행")
    for r in rows[:limit]:
        print("   ", {k: str(v)[:58] for k, v in r.items()})
    return rows


# 책 리스팅 -> 저장소 파일 대조표.
# listings/ 가 없는 챕터가 있다 (코드가 importer/·analysis/ 등에만 있는 경우).
if (HERE / "listings").is_dir():
    print(f"{'책':<8}저장소 파일")
    for path in cypher.listings(HERE / "listings"):
        print(f"{path.name.split(' ')[0]:<8}{path.name}")
else:
    print("listings/ 가 없다. 이 챕터의 코드는 아래 위치에 있다:")
    for entry in sorted(HERE.iterdir()):
        if entry.is_dir() and entry.name != "__pycache__":
            files = sorted(f.name for f in entry.rglob("*.py"))
            print(f"  {entry.name}/  {', '.join(files[:6])}")
    print()
    print("책 리스팅 번호는 원문 md 에서 확인해 study.toml 에 선언하라.")


---
### 7.1 From recognition to disambiguation — 인식에서 중의성 해소로

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


### 실습 7-A — 해설판 7.1 의 연습문제

> TODO(agent): 해설판 연습문제 지문을 그대로 인용한다.

TODO(agent): 이 실습이 절의 내용과 어떻게 이어지는지.


In [ ]:
"""실습 7-A 풀이 — TODO(agent): 무엇을 확인하는가.

TODO(agent): 책의 지문을 그대로 풀 수 없으면(데이터가 이미 그 단계를 지났다 등)
어떻게 우회했는지 밝힌다.
"""
# TODO(agent): 구현


---
### 7.2 Understanding named entity disambiguation — 개체명 중의성 해소 이해하기

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


In [ ]:
"""책 7.1, 7.2, 7.3, 7.4, 7.5 — TODO(agent): 이 코드가 무엇을 하는지 한 줄.

TODO(agent): 왜 이렇게 생겼는지. 관용구·제약·주의사항. 오래 걸리면 실측 시간을 적는다.
"""
show("7.1")
show("7.2")
show("7.3")
show("7.4")

# TODO(agent): 실제 실행. run("7.1") 형태로 호출하고 결과를 책의 표·그림과
#   대조해 출력하라. 책과 다르면 왜 다른지 적어라 (데이터 갱신 등).


---
### 7.3 Domain-based NED and LLMs — 도메인 기반 NED와 LLM

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


---
### 7.4 Business and domain understanding — 비즈니스와 도메인 이해

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


---
### 7.5 Understanding the data — 데이터 이해하기

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


In [ ]:
"""책 7.6, 7.7, 7.8, 7.9, 7.10 — TODO(agent): 이 코드가 무엇을 하는지 한 줄.

TODO(agent): 왜 이렇게 생겼는지. 관용구·제약·주의사항. 오래 걸리면 실측 시간을 적는다.
"""
show("7.6")
show("7.7")
show("7.8")
show("7.9")

# TODO(agent): 실제 실행. run("7.6") 형태로 호출하고 결과를 책의 표·그림과
#   대조해 출력하라. 책과 다르면 왜 다른지 적어라 (데이터 갱신 등).


---
### 7.6 Building a SoHO knowledge graph — SoHO 지식 그래프 구축하기

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


In [ ]:
"""책 7.11, 7.12, 7.13, 7.14, 7.15, 7.16, 7.17, 7.18, 7.19, 7.20, 7.21, 7.22, 7.23 — TODO(agent): 이 코드가 무엇을 하는지 한 줄.

TODO(agent): 왜 이렇게 생겼는지. 관용구·제약·주의사항. 오래 걸리면 실측 시간을 적는다.
"""
show("7.11")
show("7.12")
show("7.13")
show("7.14")

# TODO(agent): 실제 실행. run("7.11") 형태로 호출하고 결과를 책의 표·그림과
#   대조해 출력하라. 책과 다르면 왜 다른지 적어라 (데이터 갱신 등).


---
### 7.7 KG-based use cases — KG 기반 활용 사례

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


In [ ]:
"""책 7.24, 7.26, 7.27, 7.28, 7.29, 7.30, 7.31, 7.32, 7.33, 7.34, 7.35, 7.36, 7.37, 7.38, 7.39, 7.40 — TODO(agent): 이 코드가 무엇을 하는지 한 줄.

TODO(agent): 왜 이렇게 생겼는지. 관용구·제약·주의사항. 오래 걸리면 실측 시간을 적는다.
"""
show("7.24")
show("7.26")
show("7.27")
show("7.28")

# TODO(agent): 실제 실행. run("7.24") 형태로 호출하고 결과를 책의 표·그림과
#   대조해 출력하라. 책과 다르면 왜 다른지 적어라 (데이터 갱신 등).


---
### 7.7 KG-based use cases — KG 기반 활용 사례 (이어서)

TODO(agent): 이 절이 무엇을 하는지, 왜 이 방법인지. 해설판의 정의·수식·표를 옮기고 앞
절과의 연결을 밝힌다. 그림이 있으면 마크다운 이미지 문법으로 `attachment:` 키
`fig7-N.jpg` 를 참조하고 캡션에 무엇을 보는 그림인지 적는다 —
원격 URL 이나 상대경로가 아니라 attachment 로 내장해야 뷰어에서 보인다.


In [ ]:
"""책 7.40, 7.41 — TODO(agent): 이 코드가 무엇을 하는지 한 줄.

TODO(agent): 왜 이렇게 생겼는지. 관용구·제약·주의사항. 오래 걸리면 실측 시간을 적는다.
"""
show("7.40")
show("7.41")

# TODO(agent): 실제 실행. run("7.40") 형태로 호출하고 결과를 책의 표·그림과
#   대조해 출력하라. 책과 다르면 왜 다른지 적어라 (데이터 갱신 등).


---
## 요약

TODO(agent): 해설판 요약의 핵심을 옮기고, 이 노트북에서 실제로 본 것과 연결한다.

### 이 노트북에서 실측한 수치

| 항목 | 실측값 | 책 값 |
| --- | --- | --- |
| TODO(agent) | | |

TODO(agent): 책 값과 다르면 왜 다른지 적는다 (데이터 갱신·시드 스냅샷 등).

### 핵심 용어

| 용어 | 뜻 |
| --- | --- |
| TODO(agent) | |

### 참고 링크

- TODO(agent)

### 다음 장으로

TODO(agent): 다음 장이 방향을 어떻게 바꾸는지. 저장소 디렉터리 번호가 어긋나면 그 근거도.
